# YOLO 모델 재학습하기

YOLO 모델이 자동차 사고를 감지할 수 있도록 재학습 시키려면, 사고의 심각도 레이블이 '보통(moderate)' 또는 '심각(severe)'으로 지정된 자동차 사고 이미지 데이터셋이 필요합니다.  
우리는 RoboFlow에서 확보한 주석(annotated)이 포함된 데이터셋을 가지고 있으며, 학습용과 검증용으로 나뉘어져 있습니다.  
이 학습/검증 데이터셋을 사용하여 현재 YOLO 모델을 재학습할 것입니다.  
학습을 위해 요구되는 데이터 구조에 대한 간단한 정보는 다음과 같습니다:

1. 모델이 학습하게 될 객체 클래스는 0번: `moderate`, 1번: `severe`입니다.  
2. 데이터셋은 독립된 폴더에 저장되며, 그 안에 `train`과 `valid`라는 두 개의 하위 폴더가 포함됩니다. 각 폴더 내에는 `images`와 `labels`라는 하위 폴더가 있습니다.  
3. 각 이미지와 동일한 이름의 주석 텍스트 파일이 `labels` 폴더에 존재합니다.  
4. `data.yaml`이라는 YAML 형식의 데이터셋 설명 파일이 학습용 데이터 경로 및 클래스 정보를 포함하며, 이 파일은 모델의 `train` 메서드에 전달되어 학습 과정을 시작합니다.

그럼 시작해봅시다!

In [ ]:
# 이 실습을 위해 사전구성된 워크벤치 이미지를 사용하지 않았다면, 아래 줄의 주석을 해제하고 실행하여 필요한 패키지들을 설치할 수 있습니다.
# !pip install --no-cache-dir --no-dependencies -r requirements.txt

import os
import requests
import zipfile
from tqdm.notebook import tqdm
from ultralytics import YOLO

## os, requests, zipfile: 파일 다운로드 및 압축 해제용
## tqdm: 진행률 표시 (Notebook 전용)

이제 'yolov8m.pt' 모델을 불러오겠습니다.

In [ ]:
# 사전 학습된 모델 불러오기
model = YOLO('yolov8m.pt')

## 학습 데이터 불러오기

다음 두 가지 학습용 데이터셋(zip 파일 형식)이 제공됩니다:  
1) `accident-full.zip`   - 모델을 전체 재학습(full retraining)할 때 사용  
2) `accident-sample.zip` - 시간이 부족해 전체 재학습이 어려울 경우 부분 재학습(partial retraining)에 사용  

이번 실습에서는 `sample` 데이터셋만 사용할 예정입니다.

In [ ]:
# 특정 데이터셋을 불러오는 함수
def retrieve_dataset(dataset_type):

    # 디렉토리가 존재하는지 확인 후, 없으면 생성
    if not os.path.exists("./datasets/"):
        os.makedirs("./datasets/")

    URL = f"https://rhods-public.s3.amazonaws.com/sample-data/accident-data/accident-{dataset_type}.zip"

    # 파일이 존재하는지 확인 후, 없으면 다운로드하고 압축 해제
    if not os.path.exists(f"./datasets/accident-{dataset_type}.zip"):
        print("Downloading file...")
        response = requests.get(URL, stream=True)
        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024
        t = tqdm(total=total_size, unit='iB', unit_scale=True)
        with open(f'./datasets/accident-{dataset_type}.zip', 'wb') as f:
            for data in response.iter_content(block_size):
                t.update(len(data))
                f.write(data)
        t.close()
    if os.path.exists(f"./datasets/accident-{dataset_type}.zip"):
        print("Unzipping file...")
        with zipfile.ZipFile(f'./datasets/accident-{dataset_type}.zip', 'r') as zip_ref:
            zip_ref.extractall(path='./datasets/')
    print("Done!")


dataset_type = 'sample'
# dataset_type = 'full' # 풀 데이터셋을 가져오고 싶은 경우, 윗 라인 대신 이 라인을 사용
retrieve_dataset(dataset_type)

## YOLO 모델 재학습

우선 'epoch'의 개념을 이해해봅시다. 머신러닝 모델은 주어진 데이터셋을 알고리즘에 통과시키며 학습합니다.  
이때 전체 데이터셋이 한 번 알고리즘을 통과하면 이를 하나의 **epoch**을 완료했다고 합니다.  
각 epoch은 모델의 성능을 점차 개선해줍니다.

아래의 학습 코드에서는 다음과 같은 파라미터를 설정합니다:  
**results = model.train(data='./datasets/accident-sample/data.yaml', epochs=1, imgsz=640, batch=2)**

- `epochs`: 이 실습은 데모 목적이므로 1 epoch만 수행합니다.
- `imgsz`: 모델에 입력되는 이미지의 크기입니다.
- `batch`: 동시에 처리되는 이미지 수를 의미합니다. 많을수록 학습 성능이 좋지만 메모리도 더 필요합니다.  
  워크숍 환경은 자원이 제한되어 있으므로 `2`로 설정하였습니다.

학습이 진행되면 각 **epoch**마다 학습과 검증 결과가 출력됩니다.  
`라인 1~2`는 학습 결과, `라인 3~4`는 검증 결과입니다.

이제 아래 셀을 실행하여 모델 재학습을 시작해보세요!

In [ ]:
# 모델 학습

results = model.train(data='./datasets/accident-sample/data.yaml', epochs=1, imgsz=640, batch=2)

**참고**: 재학습 결과가 만족스럽다면, 모델을 ONNX 형식으로 내보낼 수 있습니다.  
(아래 명령어에서 `trainX`는 내보내고자 하는 학습 세션 폴더명으로 교체해야 합니다)

`ObjDetOXModel = YOLO("runs/detect/trainX/weights/best.pt").export(format="onnx")`

## 학습 결과 해석하기

**full 데이터셋**을 기반으로 한 학습 과정 및 결과에 대한 전체 설명은 실습 문서에 포함되어 있습니다.

이제 모델 재학습이 완료되었으니, 실제 사고 이미지로 테스트해봅시다!

**노트북 `04-04-accident-recog.ipynb`를 열어주세요.**